# Data Merging: SAP.csv and FM down.csv

This notebook loads SAP.csv into a dataframe and adds EquipmentGuid from FM down.csv based on:
1. Matching Parent_floc (SAP) to AttributeValue (FM down)
2. Text similarity between equip_desc (SAP) and Name (FM down)


In [ ]:
import pandas as pd
import numpy as np
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')


In [ ]:
def calculate_similarity(text1, text2):
    """Calculate similarity between two text strings using SequenceMatcher"""
    if pd.isna(text1) or pd.isna(text2):
        return 0
    return SequenceMatcher(None, str(text1).lower(), str(text2).lower()).ratio() * 100


In [ ]:
def clean_equip_desc(equip_desc, parent_location=None):
    """Clean equipment description by replacing abbreviations with full words and removing parent location words"""
    # Handle None/NaN values
    if pd.isna(equip_desc) or equip_desc is None:
        return equip_desc
    
    # Define abbreviation mappings
    abbreviations = {
        'pmp': 'pump',
        'metr': 'meter', 
        'tk': 'tank',
        'cmpr': 'compressor',
        'wtr': 'water',
        'sls': 'sales',
        'sep': 'separator',
        
    }
    
    try:
        # Convert to string and split into words
        words = str(equip_desc).strip().split()
        
        # If empty, return as is
        if not words:
            return equip_desc
        
        # Get parent location words to remove (if provided)
        parent_words = set()
        if parent_location and not pd.isna(parent_location):
            parent_words = set(str(parent_location).strip().lower().split())
        
        # Clean words: replace abbreviations and remove parent location words
        cleaned_words = []
        for word in words:
            word_lower = word.lower()
            
            # Skip if word is in parent location
            if word_lower in parent_words:
                continue
            
            # Replace abbreviations
            if word_lower in abbreviations:
                cleaned_words.append(abbreviations[word_lower])
            else:
                cleaned_words.append(word)
        
        return ' '.join(cleaned_words)
        
    except Exception as e:
        # If any error occurs, return the original description
        print(f"Error cleaning '{equip_desc}': {e}")
        return equip_desc


In [ ]:
# Load SAP.csv
print("Loading SAP.csv...")
sap_df = pd.read_csv('SAP.csv')
print(f"SAP dataframe shape: {sap_df.shape}")
print(f"SAP columns: {list(sap_df.columns)}")
print("\nFirst few rows of SAP data:")
sap_df['Equip_desc_cleaned'] = sap_df.apply(lambda row: clean_equip_desc(row['Equip_desc'], row['Parent_Location']), axis=1)
sap_df.head()


In [ ]:
# Load FM down.csv
print("Loading FM down.csv...")
fm_df = pd.read_csv('FM down.csv')
print(f"FM down dataframe shape: {fm_df.shape}")
print(f"FM down columns: {list(fm_df.columns)}")
print("\nFirst few rows of FM down data:")
fm_df.head()


In [ ]:
# Step 1: Direct matching on Parent_floc = AttributeValue
print("Step 1: Direct matching on Parent_floc = AttributeValue")
direct_matches = sap_df.merge(
    fm_df, 
    left_on='Parent_floc', 
    right_on='AttributeValue', 
    how='left',
    suffixes=('_sap', '_fm')
)

print(f"Direct matches found: {direct_matches['EquipmentGuid'].notna().sum()}")
print(f"Total SAP records: {len(sap_df)}")
print(f"Match rate: {direct_matches['EquipmentGuid'].notna().sum() / len(sap_df) * 100:.2f}%")


In [ ]:
print(direct_matches)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def clean_dataframe(direct_matches):
    """
    Create one-to-one mapping between FLOC_ID and EquipmentGuid
    Each FLOC_ID gets exactly one EquipmentGuid match
    """
    result_df = pd.DataFrame()
    used_equipment_guids = set()  # Track which EquipmentGuids have been used
    used_floc_ids = set()  # Track which FLOC_IDs have been matched
    
    # Define equipment type words that must match between equip_desc_cleaned and name
    equipment_type_words = {'pump', 'meter', 'tank', 'compressor', 'separator'}
    
    def validate_equipment_type_match(equip_desc_cleaned, name):
        """Check if equipment type words in equip_desc_cleaned also exist in name"""
        if pd.isna(equip_desc_cleaned) or pd.isna(name):
            return True  # Allow matches if either is missing
        
        equip_words = set(str(equip_desc_cleaned).lower().split())
        name_words = set(str(name).lower().split())
        
        # Find equipment type words in equip_desc_cleaned
        equip_type_words_found = equip_words.intersection(equipment_type_words)
        
        # If no equipment type words found in equip_desc_cleaned, allow the match
        if not equip_type_words_found:
            return True
        
        # Check if all equipment type words from equip_desc_cleaned exist in name
        return equip_type_words_found.issubset(name_words)
    
    # Group by FLOC_ID
    for floc_id, group in direct_matches.groupby('FLOC_ID'):
        # Skip if this FLOC_ID has already been matched
        if floc_id in used_floc_ids:
            continue
            
        # Filter out already used EquipmentGuids
        available_group = group[~group['EquipmentGuid'].isin(used_equipment_guids)]
        
        # If no available EquipmentGuids for this FLOC_ID, skip
        if len(available_group) == 0:
            continue
            
        # If only one row for this FLOC_ID, validate equipment type match
        if len(available_group) == 1:
            row = available_group.iloc[0]
            if validate_equipment_type_match(row['Equip_desc_cleaned'], row['Name']):
                result_df = pd.concat([result_df, available_group])
                used_equipment_guids.add(row['EquipmentGuid'])
                used_floc_ids.add(floc_id)
            continue
        
        # For multiple rows with same FLOC_ID, find best match
        names = available_group['Name'].tolist()
        equip_descs = available_group['Equip_desc'].tolist()
        equip_descs_cleaned = available_group['Equip_desc_cleaned'].tolist()
        
        # Ensure we have the same number of names and equipment descriptions
        if len(names) != len(equip_descs) or len(names) != len(equip_descs_cleaned):
            print(f"Warning: Mismatch in names ({len(names)}), equip_descs ({len(equip_descs)}), or equip_descs_cleaned ({len(equip_descs_cleaned)}) for FLOC_ID {floc_id}")
            # Use the minimum length to avoid index errors
            min_length = min(len(names), len(equip_descs), len(equip_descs_cleaned))
            names = names[:min_length]
            equip_descs = equip_descs[:min_length]
            equip_descs_cleaned = equip_descs_cleaned[:min_length]
        
        # Skip if no valid data
        if len(names) == 0 or len(equip_descs) == 0:
            continue
            
        # Create TF-IDF vectorizer for text comparison
        vectorizer = TfidfVectorizer(stop_words='english')
        
        # Combine all text for vectorization
        all_text = names + equip_descs
        
        try:
            tfidf_matrix = vectorizer.fit_transform(all_text)
            
            # Verify matrix dimensions
            expected_rows = len(names) + len(equip_descs)
            actual_rows = tfidf_matrix.shape[0]
            
            if actual_rows != expected_rows:
                print(f"Warning: TF-IDF matrix has {actual_rows} rows, expected {expected_rows}")
                continue
            
            # Calculate similarity between each Name and Equip_desc pair
            similarities = []
            valid_matches = []  # Track which matches pass equipment type validation
            
            for i in range(len(names)):
                name_vector = tfidf_matrix[i]
                equip_vector = tfidf_matrix[i + len(names)]
                similarity = cosine_similarity(name_vector, equip_vector)[0][0]
                similarities.append(similarity)
                
                # Check equipment type validation
                is_valid = validate_equipment_type_match(equip_descs_cleaned[i], names[i])
                valid_matches.append(is_valid)
                
        except Exception as e:
            print(f"Error processing FLOC_ID {floc_id}: {e}")
            continue
        
        # Find valid matches with highest similarity
        valid_indices = [i for i, is_valid in enumerate(valid_matches) if is_valid]
        
        if not valid_indices:
            # No valid equipment type matches found, skip this group
            continue
        
        # Among valid matches, find the one with highest similarity
        valid_similarities = [similarities[i] for i in valid_indices]
        best_valid_idx = valid_indices[np.argmax(valid_similarities)]
        best_similarity = similarities[best_valid_idx]
        
        # Only add the match if similarity is greater than 0.02
        if best_similarity > 0.02:
            # Add the best match row to the result
            best_match_row = available_group.iloc[[best_valid_idx]].copy()
            best_match_row['similarity_score'] = best_similarity
            result_df = pd.concat([result_df, best_match_row])
            
            # Mark this EquipmentGuid and FLOC_ID as used
            used_equipment_guids.add(best_match_row.iloc[0]['EquipmentGuid'])
            used_floc_ids.add(floc_id)
        # If similarity is <= 0.02, skip this group (no match will be added)
    
    return result_df

# Apply the function to clean the dataframe
cleaned_df = clean_dataframe(direct_matches)

# Reset index for the final dataframe
cleaned_df = cleaned_df.reset_index(drop=True)

# Display the result
print(f"Original dataframe had {len(direct_matches)} rows")
print(f"Cleaned dataframe has {len(cleaned_df)} rows")
print(f"Unique EquipmentGuids used: {len(cleaned_df['EquipmentGuid'].unique())}")
cleaned_df.head()

In [ ]:
result_df2 = cleaned_df
result_df2.to_csv('merged_data2.csv', index=False)

In [ ]:
# Final results
print("FINAL RESULTS:")
print(f"Original SAP records: {len(sap_df)}")
print(f"Final merged records: {len(direct_matches)}")
print(f"Records with EquipmentGuid: {direct_matches['EquipmentGuid'].notna().sum()}")
print(f"Final match rate: {direct_matches['EquipmentGuid'].notna().sum() / len(direct_matches) * 100:.2f}%")

# Verify row count consistency
if len(direct_matches) == len(sap_df):
    print("✓ SUCCESS: Final dataframe has same number of rows as original SAP dataframe")
else:
    print("✗ WARNING: Row count mismatch detected!")

# Show some examples of matches
print("\nSample matches:")
matched_samples = direct_matches[direct_matches['EquipmentGuid'].notna()].head(10)
for idx, row in matched_samples.iterrows():
    print(f"SAP: {row['Equip_desc']} | FM: {row['Name']} | Parent_floc: {row['Parent_floc']} | AttributeValue: {row['AttributeValue']}")
